# 35 · Sample satellite files at points, checked against h5py and pyproj

## What this checks

**Sample satellite files at points or over areas** reads a Sentinel-5P satellite file in the browser. It is now
retired. This check covers release 1.0.0, its last.

For each point and each pass of the satellite, it wrote one row. The row held the value of the pixel whose centre is
nearest the point, its quality, and, where there was no usable value, which kind of gap it was. The New York City
worked example read the four NO₂ monitors this way. A point given the wrong pixel gets another place's air.

## How it is checked

We read the same file with **h5py**, a Python reader for HDF5, the file format these satellite files use. Fieldwork
reads it with h5wasm instead. The two are written by different authors.

Both rest on the same underlying HDF5 library, so a fault in that library would be shared. What differs is
everything Fieldwork wrote itself: which parts of the file it reads, in what order, when it stops looking, and which
pixel it keeps. Fieldwork reads the file in blocks, called chunks. Here we read the whole swath, the strip the
satellite sees, at once, and compare every pixel centre. So a point that Fieldwork's search places on the wrong
pixel, or stops looking for too early, shows up.

Which pixel is nearest is Fieldwork's own stated rule, so we restate it here. The rule measures distance in degrees,
with the longitude scaled by the cosine of the point's latitude. That is an approximation. So we also find the truly
nearest centre on the curved Earth with **pyproj**, and report it beside the rule. That is a measure of the
approximation, not part of the criteria.

The data are a made-up file, shaped like the real NO₂ file, with no Copernicus data in it. It has 13 points on a swath
of 12 × 10 pixels, stored in 3 × 2 chunks.

## What should happen

1. Every point is placed, or not placed, as the stated rule says, on the same pixel.
2. At every placed pixel, the value, the stored quality value and the time are exactly what h5py finds there.
3. Fieldwork can keep pixels with a quality value of at least 0.75, or above 0.75. For both, every row of the table
   is rebuilt from the stated rule, field by field. The rows, their order, the gap kind, the quality outcome and the
   value must all be the same.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["numpy", "h5py", "pyproj"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_35_sample_satellite import run

report = run(Path.cwd() / "fixtures" / "sample-satellite-s5p-001.json")
print("agrees:", report["agrees"])
for x in report["placement"]:
    print(f"{x['point']:<16} fieldwork {x['fieldwork']}  declared rule {x['declaredRule']}  "
          f"{'same' if x['samePixel'] and x['valueQualityTimeAgree'] else 'DIFFERS'}")
g = report["geodesic"]
print(f"geodesic nearest = declared rule at {g['same']} of {g['of']} placed points")
for c in report["cases"]:
    print(f"qa_value {c['operator']} 0.75: {c['rows']['fieldwork']} rows; {c['counts']['fieldwork']}; "
          f"{'agrees' if c['agrees'] else 'differs at ' + str(c['rowsDiffering'])}")


## The result on record

First recorded on 6 November 2026, against release 0.2.1. Rerun on 26 November 2026, against release 1.0.0:
**Agrees**.

- **Placing.** 11 points were placed and 2 fell off the swath. Every pixel is the one the stated rule gives.
- **Values.** At each, the value, quality and time are exactly what h5py reads.
- **The approximation.** The truly nearest centre, from pyproj, is the same pixel at 11 of 11 placed points.
- **The table.** For at least 0.75 and above 0.75, each of the 26 rows was rebuilt field by field. 7 and 6 rows
  pass quality.

The check found a defect before it agreed. In release 0.2.0, a point near the edge of a chunk got the pixel across
that edge. It got the nearest pixel within the first chunk, not within the whole swath. Release 0.2.1 fixed it on
6 November 2026.

The notebook also agrees when run in the browser.

Running the cell above repeats the check on the same data.

## What this does not show

- Anything about a real satellite file's content. The file here is made up.
- Reading a file over the network piece by piece, which is where the time goes.
- Whether the nearest centre is the right pixel for a study.
- A fault in the HDF5 library itself, which h5py and h5wasm share.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture:", report["fixture"]["id"], report["fixture"]["operation"]["widget"], report["fixture"]["operation"]["release"])
